# MLB Fan Pulse: baseline sentiment (Phase 4)

Scores every comment with `cardiffnlp/twitter-roberta-base-sentiment-latest` and writes the `comment_scores` table.

1. **Runtime > Change runtime type > T4 GPU** (it also works on CPU, just slower).
2. **Runtime > Run all.**
3. When asked, upload `data/processed/comments/<game_pk>.parquet` from the project.
4. The result downloads as `<game_pk>.parquet`. Move it to `data/processed/comment_scores/` in the project (create the folder if needed), then run `mlb-fan-pulse sentiment --game <game_pk>` locally.

The scoring code mirrors `src/mlb_fan_pulse/process/sentiment.py`. Keep the two in step.

In [ ]:
import re

import pandas as pd
import torch
from google.colab import files
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
MAX_TOKENS = 256
BATCH_SIZE = 64

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
uploaded = files.upload()
filename = next(iter(uploaded))
comments = pd.read_parquet(filename)
print(f"{filename}: {len(comments)} comments")
comments.head()

In [ ]:
MEDIA_EMBED = re.compile(r"!\[(?:gif|img)\]\([^)]*\)")
URL = re.compile(r"https?://\S+")
USER = re.compile(r"(?<!\w)/?u/[\w-]+")


def preprocess(text):
    text = MEDIA_EMBED.sub("", text)
    text = URL.sub("http", text)
    text = USER.sub("@user", text)
    return " ".join(text.split())


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device).eval()
labels = [model.config.id2label[i].lower() for i in range(model.config.num_labels)]
print("labels:", labels)

cleaned = [preprocess(t) for t in comments["body"].fillna("")]
# Similar lengths together keeps padding (and so runtime) down.
order = sorted(range(len(cleaned)), key=lambda i: len(cleaned[i]))
probabilities = [None] * len(cleaned)

with torch.no_grad():
    for start in range(0, len(order), BATCH_SIZE):
        indices = order[start : start + BATCH_SIZE]
        encoded = tokenizer(
            [cleaned[i] for i in indices], padding=True, truncation=True, max_length=MAX_TOKENS, return_tensors="pt"
        ).to(device)
        batch = torch.softmax(model(**encoded).logits, dim=-1).cpu().tolist()
        for i, row in zip(indices, batch):
            probabilities[i] = row
        if (start // BATCH_SIZE) % 20 == 0:
            print(f"{min(start + BATCH_SIZE, len(order))}/{len(order)}")

scores = pd.DataFrame(probabilities, columns=[f"base_{label[:3]}" for label in labels])
scores["base_sentiment"] = scores["base_pos"] - scores["base_neg"]
scores.insert(0, "comment_id", comments["comment_id"].values)
scores = scores[["comment_id", "base_sentiment", "base_neg", "base_neu", "base_pos"]]
scores.describe()

In [ ]:
# Eyeball check: the most negative and most positive comments.
check = comments[["comment_id", "body"]].merge(scores, on="comment_id").sort_values("base_sentiment")
pd.set_option("display.max_colwidth", 160)
display(check[["base_sentiment", "body"]].head(10))
display(check[["base_sentiment", "body"]].tail(10))

In [ ]:
assert len(scores) == len(comments) and scores["base_sentiment"].notna().all()
scores.to_parquet(filename, index=False)  # same <game_pk>.parquet name, now holding the scores
files.download(filename)